## PRÁCTICA 2: Aprendizaje Semi-supervisado y en Una Clase

### Cargar dataset

##### Trabajaremos con el conjunto de datos CIFAR-100 (50.000 instancias para entrenamiento y 10.000 para test), en el cual eliminaremos el 80% de sus etiquetas y realizaremos las siguientes particiones:

#####    40.000 instancias de entrenamiento no etiquetadas
#####    10.000 instancias de entrenamiento etiquetadas
#####    10.000 instancias de test etiquetadas

In [1]:
import tensorflow as tf
import numpy as np

In [2]:
labeled_data = 0.2 # Vamos a usar el etiquetado de sólo el 20% de los datos
np.random.seed(42)

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar100.load_data()

indexes = np.arange(len(x_train))
np.random.shuffle(indexes)
ntrain_data = int(labeled_data*len(x_train))
unlabeled_train = x_train[indexes[ntrain_data:]]
x_train = x_train[indexes[:ntrain_data]]
y_train = y_train[indexes[:ntrain_data]]

In [3]:
print(x_train.shape)
print(unlabeled_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)

(10000, 32, 32, 3)
(40000, 32, 32, 3)
(10000, 1)
(10000, 32, 32, 3)
(10000, 1)


#### Preprocesado de los datos

In [4]:
# convertimos de uint8 a float32 y normalizamos en rango [0, 1]
x_train = x_train.reshape(-1, 32, 32, 3).astype("float32") / 255.0
x_test = x_test.reshape(-1, 32, 32, 3).astype("float32") / 255.0
unlabeled_train = unlabeled_train.reshape(-1, 32, 32, 3).astype("float32") / 255.0


#### 1. Entrena un modelo, creado sobre TensorFlow, haciendo uso únicamente de las instancias etiquetadas de entrenamiento. Dicho modelo debe de tener al menos cuatro capas densas y/o convolucionales.

In [ ]:
miCNN = tf.keras.models.Sequential([
        tf.keras.layers.Conv2D(64, (3, 3), activation="relu", padding='same', input_shape=(32, 32, 3)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Conv2D(64, (3, 3), activation="relu", padding='same'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
        tf.keras.layers.Dropout(0.2),

        tf.keras.layers.Conv2D(128, (3, 3), activation="relu", padding='same'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Conv2D(128, (3, 3), activation="relu", padding='same'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
        tf.keras.layers.Dropout(0.3),

        tf.keras.layers.Conv2D(256, (3, 3), activation="relu", padding='same'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
        tf.keras.layers.Dropout(0.4),

        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(512, activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(100, activation="softmax")
    ])

optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

miCNN.compile(
            loss="sparse_categorical_crossentropy",
            optimizer=optimizer,
            metrics=["accuracy"]
        )

miCNN.fit(x_train, y_train, epochs=50, batch_size=64, verbose=1)

_, test_acc = miCNN.evaluate(x_test, y_test, verbose=2)

print("Test accuracy: ", test_acc)


#### Responde las siguientes preguntas:
####    a. ¿Qué red has escogido? ¿Por qué? ¿Cómo la has entrenado?
####    b. ¿Cuál es el rendimiento del modelo en entrenamiento? ¿Y en prueba?
####    c. ¿Qué conclusiones sacas de los resultados detallados en el punto anterior?

#### 2. Entrena el mismo modelo, incorporando las instancias no etiquetadas de entrenamiento mediante la técnica de auto-aprendizaje. Opcionalmente, se ponderará cada instancia de entrada en función de su calidad (o certeza).

In [ ]:
def miCNN_func():
    miCNN = tf.keras.models.Sequential([
            tf.keras.layers.Conv2D(64, (3, 3), activation="relu", padding='same', input_shape=(32, 32, 3)),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.Conv2D(64, (3, 3), activation="relu", padding='same'),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
            tf.keras.layers.Dropout(0.2),

            tf.keras.layers.Conv2D(128, (3, 3), activation="relu", padding='same'),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.Conv2D(128, (3, 3), activation="relu", padding='same'),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
            tf.keras.layers.Dropout(0.3),

            tf.keras.layers.Conv2D(256, (3, 3), activation="relu", padding='same'),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
            tf.keras.layers.Dropout(0.4),

            tf.keras.layers.Flatten(),
            tf.keras.layers.Dense(512, activation="relu"),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.Dropout(0.5),
            tf.keras.layers.Dense(100, activation="softmax")
        ])

    optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

    miCNN.compile(
                loss="sparse_categorical_crossentropy",
                optimizer=optimizer,
                metrics=["accuracy"]
            )
    
    return miCNN

In [ ]:
def self_training(model_func, x_train, y_train, unlabeled_data, x_test, y_test, thresh=0.5, train_epochs=3):
    train_data = x_train.copy()
    train_label = y_train.copy()

    for i in range(train_epochs):
        model = model_func()
        print("Train Epoch: ", (i+1))
        model.fit(train_data, train_label, epochs=20,
                  batch_size=128, verbose=1)

        y_pred = model.predict(unlabeled_data)

        y_class = np.argmax(y_pred, axis=1)
        y_value = np.max(y_pred, axis=1)

        train_data = x_train.copy()
        train_label = y_train.copy()

        for x_u, y_c, y_v in zip(unlabeled_data, y_class, y_value):
            if y_v > thresh:
                x_u = x_u[np.newaxis, ...]   # pasamos de (32,32,3) a (1,32,32,3)
                train_data = np.concatenate((train_data, x_u), axis=0)
                train_label = np.append(train_label, y_c)

    return model

In [ ]:
model = self_training(
    miCNN_func,
    x_train[::5],
    y_train[::5],
    unlabeled_train[::5],
    x_test,
    y_test,
    thresh=0.9,
    train_epochs=5
)

_, test_accuracy = model.evaluate(x_test, y_test, verbose=1)

print("Test accuracy: ", test_accuracy)

#### Responde las siguientes preguntas:
####    a. ¿Qué parámetros has definido para el entrenamiento?
####    b. ¿Cuál es el rendimiento del modelo en entrenamiento? ¿Y en prueba?
####    c. ¿Se mejoran los resultados obtenidos en el Ejercicio 1?
####    d. ¿Qué conclusiones sacas de los resultados detallados en los puntos anteriores?

#### 3. Entrena un modelo de aprendizaje semisupervisado de tipo autoencoder en dos pasos (primero el autoencoder, después el clasificador). La arquitectura del encoder debe ser exactamente la misma que la definida en los Ejercicios 1 y 2, a excepción del último bloque de capas.

In [ ]:
class MiAutoencoder:
    def __init__(self, input_shape):
        # encoder
        input_layer = tf.keras.layers.Input(shape=input_shape)
        x = tf.keras.layers.Conv2D(64, (3, 3), activation="relu", padding='same')(input_layer)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Conv2D(64, (3, 3), activation="relu", padding='same')(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(x)
        x = tf.keras.layers.Dropout(0.2)(x)

        x = tf.keras.layers.Conv2D(128, (3, 3), activation="relu", padding='same')(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Conv2D(128, (3, 3), activation="relu", padding='same')(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(x)
        x = tf.keras.layers.Dropout(0.3)(x)

        x = tf.keras.layers.Conv2D(256, (3, 3), activation="relu", padding='same')(x)
        x = tf.keras.layers.BatchNormalization()(x)
        code = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(x)
        
        # decoder
        x = tf.keras.layers.UpSampling2D((2, 2))(code)
        x = tf.keras.layers.Conv2D(256, (3, 3), activation="relu", padding='same')(x)
        x = tf.keras.layers.BatchNormalization()(x)

        x = tf.keras.layers.UpSampling2D((2, 2))(x)
        x = tf.keras.layers.Conv2D(128, (3, 3), activation="relu", padding='same')(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Conv2D(128, (3, 3), activation="relu", padding='same')(x)
        x = tf.keras.layers.BatchNormalization()(x)

        x = tf.keras.layers.UpSampling2D((2, 2))(x)
        x = tf.keras.layers.Conv2D(64, (3, 3), activation="relu", padding='same')(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Conv2D(64, (3, 3), activation="relu", padding='same')(x)
        x = tf.keras.layers.BatchNormalization()(x)
        output_layer = tf.keras.layers.Conv2D(3, (3, 3), activation="sigmoid", padding="same")(x)

        self.autoencoder = tf.keras.Model(input_layer, output_layer)
        self.encoder = tf.keras.Model(input_layer, code)
        self.decoder = tf.keras.Model(code, output_layer)

        self.autoencoder.compile(optimizer="adam", loss="mse")

    def fit(self, X, y=None, sample_weight=None):
        self.autoencoder.fit(
            X, X,
            epochs=5,
            batch_size=128,
            shuffle=True,
            sample_weight=sample_weight,
            verbose=1
        )

    def get_encoded_data(self, X):
        return self.encoder.predict(X)

    def __del__(self):
        tf.keras.backend.clear_session()

In [ ]:
class MiClasificador:

    def __init__(self, input_shape):
        self.model = tf.keras.Sequential([
            tf.keras.layers.Flatten(input_shape=(4, 4, 256)),
            tf.keras.layers.Dense(512, activation="relu"),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.Dropout(0.5),
            tf.keras.layers.Dense(100, activation="softmax")
        ])

        self.model.compile(
            optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

    def fit(self, X, y, sample_weight=None):
        self.model.fit(
            X, y,
            epochs=10,
            batch_size=128,
            shuffle=True,
            sample_weight=sample_weight,
            verbose=1
        )

    def predict(self, X):
        probs = self.model.predict(X)
        return np.argmax(probs, axis=1)

    def predict_proba(self, X):
        return self.model.predict(X)

    def score(self, X, y):
        loss, acc = self.model.evaluate(X, y, verbose=0)
        return acc

    def __del__(self):
        tf.keras.backend.clear_session()

In [ ]:
def semisupervised_training(autoencoder, classifier, x_train, y_train, unlabeled_data):

    all_data = np.concatenate((x_train, unlabeled_data), axis=0)
    autoencoder.fit(all_data)
    code = autoencoder.get_encoded_data(x_train)
    classifier.fit(code, y_train)

    return autoencoder, classifier

In [ ]:
autoencoder = MiAutoencoder(input_shape=x_train.shape[1:])
classifier = MiClasificador(input_shape=256)  # tamaño del code

autoencoder, classifier = semisupervised_training(
    autoencoder,
    classifier,
    x_train,
    y_train,
    unlabeled_train
)

pred_data = autoencoder.get_encoded_data(x_test)
print("Test accuracy :", classifier.score(pred_data, y_test))

#### Responde las siguientes preguntas:
####    a. ¿Cuál es la arquitectura del modelo? ¿Y sus hiperparámetros?
####    b. ¿Cuál es el rendimiento del modelo en entrenamiento? ¿Y en prueba?
####    c. ¿Se mejoran los resultados obtenidos en los Ejercicios 1 y 2?
####    d. ¿Qué conclusiones sacas de los resultados detallados en los puntos anteriores?

#### 4. Entrena un modelo de aprendizaje semisupervisado de tipo autoencoder en un paso (autoencoder y clasificador al mismo tiempo). La arquitectura del autoencoder será la misma que la definida en el Ejercicio 3, y la combinación de encoder y clasificador será igual a la arquitectura definida en el Ejercicio 1.

In [ ]:
class MiClasificadorSemisupervisado:
    def __init__(self, input_shape):
        inputs = tf.keras.layers.Input(shape=input_shape)
        # encoder
        x = tf.keras.layers.Conv2D(64, (3, 3), activation="relu", padding='same')(inputs)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Conv2D(64, (3, 3), activation="relu", padding='same')(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(x)
        x = tf.keras.layers.Dropout(0.2)(x)

        x = tf.keras.layers.Conv2D(128, (3, 3), activation="relu", padding='same')(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Conv2D(128, (3, 3), activation="relu", padding='same')(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(x)
        x = tf.keras.layers.Dropout(0.3)(x)

        x = tf.keras.layers.Conv2D(256, (3, 3), activation="relu", padding='same')(x)
        x = tf.keras.layers.BatchNormalization()(x)
        code = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(x)
        
        # decoder
        x_decoder = tf.keras.layers.UpSampling2D((2, 2))(code)
        x_decoder = tf.keras.layers.Conv2D(256, (3, 3), activation="relu", padding='same')(x_decoder)
        x_decoder = tf.keras.layers.BatchNormalization()(x_decoder)

        x_decoder = tf.keras.layers.UpSampling2D((2, 2))(x_decoder)
        x_decoder = tf.keras.layers.Conv2D(128, (3, 3), activation="relu", padding='same')(x_decoder)
        x_decoder = tf.keras.layers.BatchNormalization()(x_decoder)
        x_decoder = tf.keras.layers.Conv2D(128, (3, 3), activation="relu", padding='same')(x_decoder)
        x_decoder = tf.keras.layers.BatchNormalization()(x_decoder)

        x_decoder = tf.keras.layers.UpSampling2D((2, 2))(x_decoder)
        x_decoder = tf.keras.layers.Conv2D(64, (3, 3), activation="relu", padding='same')(x_decoder)
        x_decoder = tf.keras.layers.BatchNormalization()(x_decoder)
        x_decoder = tf.keras.layers.Conv2D(64, (3, 3), activation="relu", padding='same')(x_decoder)
        x_decoder = tf.keras.layers.BatchNormalization()(x_decoder)
        decoding = tf.keras.layers.Conv2D(3, (3, 3), activation="sigmoid", padding="same", name="decoding")(x_decoder)

        # clasificador
        x_classifier = tf.keras.layers.Flatten()(code)
        x_classifier = tf.keras.layers.Dense(512, activation="relu")(x_classifier)
        x_classifier = tf.keras.layers.BatchNormalization()(x_classifier)
        x_classifier = tf.keras.layers.Dropout(0.5)(x_classifier)
        classification = tf.keras.layers.Dense(100, activation="softmax", name="classification")(x_classifier)

        self.model = tf.keras.Model(inputs=inputs, outputs=[decoding, classification])

        self.model.compile(
            optimizer="adam",
            loss={"decoding": "mse", "classification": "sparse_categorical_crossentropy"},
            metrics={"classification": "accuracy"}
        )

    def fit(self, X, y, unlabeled_data):
        X_all = np.concatenate((X, unlabeled_data), axis=0)

        y_decoding = X_all    # la etiqueta es la propia entrada
        y_classification = np.concatenate((y.reshape(-1), np.zeros(len(unlabeled_data))))

        w_classification = np.concatenate((np.ones(len(X)), np.zeros(len(unlabeled_data))))

        self.model.fit(
            X_all,
            [y_decoding, y_classification],
            sample_weight=[None, w_classification],
            epochs=20,
            batch_size=256,
            shuffle=True,
            verbose=1
        )

    def predict(self, X):
        _, probs = self.model.predict(X)
        return np.argmax(probs, axis=1)

    def predict_proba(self, X):
        _, probs = self.model.predict(X)
        return probs

    def score(self, X, y):
        preds = self.predict(X)
        return np.mean(preds == y)

    def __del__(self):
        tf.keras.backend.clear_session()

In [ ]:
def semisupervised_training_v2(model, x_train, y_train, unlabeled_data):
    model.fit(x_train, y_train, unlabeled_data)
    return model

In [ ]:
model = MiClasificadorSemisupervisado(input_shape=x_train.shape[1:])

model = semisupervised_training_v2(
    model,
    x_train,
    y_train,
    unlabeled_train
)

print("Test accuracy :", model.score(x_test, y_test))

#### Responde las siguientes preguntas:
####    a. ¿Cuál es la arquitectura del modelo? ¿Y sus hiperparámetros?
####    b. ¿Cuál es el rendimiento del modelo en entrenamiento? ¿Y en prueba?
####    c. ¿Se mejoran los resultados obtenidos en los ejercicios anteriores?
####    d. ¿Qué conclusiones sacas de los resultados detallados en los puntos anteriores?

#### 5. Repite el mismo entrenamiento de los Ejercicios 2-4, pero eliminando las instancias no etiquetadas más atípicas con respecto a los datos etiquetados. Se cumplirán los siguientes puntos:

#### · La arquitectura de la red de clasificación en una clase será la misma a la utilizada en el clasificador del Ejercicio 1, a excepción de la capa de salida.

#### ·  Utiliza la técnica explicada en el Notebook 5, usando un valor de 𝑣 = 0,9.

In [5]:
class ChangeRCallback(tf.keras.callbacks.Callback):
   def __init__(self, train_data, delta=.025, steps=3):
       super().__init__()
       self.train_data = train_data
       self.delta = delta
       self.steps = steps
       self.cont = 0

   def on_epoch_end(self, epoch, logs=None):
       sorted_values = np.sort(self.model.predict(self.train_data).flatten())
       new_value = sorted_values[int(len(sorted_values) * (1. - self.model.nu))]
       old_value = self.model.r.numpy()
       print('Cambiando r a', new_value, ', max:', sorted_values.max(), ', min:', sorted_values.min())
       self.model.r.assign(new_value)
       if np.abs(old_value - new_value) < self.delta:
            self.cont += 1
            if self.cont >= self.steps:
                print('Convergencia obtenida. Finalizando el entrenamiento.')
                self.model.stop_training = True
       else:
            self.cont = 0

In [11]:
class DetectorAnomalias:

    def __init__(self, input_shape, nu=.5):
        l2_val = 0.01
        input_shape = (input_shape, 32, 3)
        inputs = tf.keras.layers.Input(shape=input_shape)
        x = tf.keras.layers.Conv2D(64, 3, activation='relu', kernel_regularizer=tf.keras.regularizers.L2(l2_val))(inputs)
        x = tf.keras.layers.MaxPooling2D(2)(x)
        x = tf.keras.layers.Conv2D(32, 3, activation='relu', kernel_regularizer=tf.keras.regularizers.L2(l2_val))(x)
        x = tf.keras.layers.MaxPooling2D(2)(x)
        x = tf.keras.layers.Flatten()(x)
        deep_features = tf.keras.layers.Dense(16, activation='relu', kernel_regularizer=tf.keras.regularizers.L2(l2_val))(x)
        outputs = tf.keras.layers.Dense(1, activation='linear', kernel_regularizer=tf.keras.regularizers.L2(l2_val))(deep_features)

        self.encoder = tf.keras.models.Model(inputs, deep_features)
        self.model = tf.keras.models.Model(inputs, outputs)

        self.model.r = tf.Variable(1.0, trainable=False, name='r', dtype=tf.float32)
        self.model.nu = tf.Variable(nu, trainable=False, name='nu', dtype=tf.float32)

        self.model.compile(optimizer="Adam",
                           loss=self.loss_function)

    def loss_function(self, y_true, y_pred):
        l2 = 0.0
        for var in self.model.trainable_weights:
            l2 += tf.reduce_sum(tf.square(var))

        reg_loss = 0.5 * l2
        return reg_loss + (1 / self.model.nu * tf.cast(tf.shape(y_true)[0], tf.float32)) * tf.reduce_sum(tf.maximum(0.0, self.model.r - y_pred))
    
    def fit(self, X, y=None, sample_weight=None):
        dummy_y = np.zeros((len(X), 1))

        return self.model.fit(
            X,
            dummy_y,
            epochs=20,
            batch_size=32,
            verbose=1,
            callbacks=[
                ChangeRCallback(X)
            ]
        )
        
    def predict(self, X, batch_size=32):
        preds = []
        for i in range(0, len(X), batch_size):
            batch = X[i:i+batch_size]
            p = self.model(batch, training=False)
            preds.append(p.numpy())
        return np.vstack(preds)
    
    def get_encoded_data(self, X):
        return self.encoder(X)
        
    def __del__(self):
        tf.keras.backend.clear_session()

In [13]:
modelo_detector = DetectorAnomalias(32, 0.9)
modelo_detector.fit(x_train)



Epoch 1/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Cambiando r a 5.7670207 , max: 13.595607 , min: 1.9873577
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 27.1108
Epoch 2/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Cambiando r a 18.163546 , max: 42.801945 , min: 6.277295
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 11.9235
Epoch 3/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Cambiando r a 141.40486 , max: 335.37357 , min: 47.73124
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 13.1698
Epoch 4/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Cambiando r a 684.7814 , max: 1631.4696 , min: 226.94362
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 27.1998
Epoch 5/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Cambiando r a 5058.9893 , max: 12114.002 , min: 1661.4684
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 140.1492
Epoch 6/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Cambiando r a 21677.822 , max: 52070.883 , min: 7038.674
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 502.0925 

In [20]:
np.count_nonzero(modelo_detector.predict(unlabeled_train) > modelo_detector.model.r)

np.int64(35965)

#### Responde las siguientes preguntas:
####    a. ¿Se mejoran los resultados con respecto a los anteriores ejercicios? ¿Qué conclusiones sacas de estos resultados?

#### 6. Repite los Ejercicios 3-5 cambiando el autencoder por la técnica definida en el apartado “Hay vida más allá del autoencoder” del Notebook 4. Contesta a las preguntas de dichos ejercicios. Se cumplirán los siguientes puntos:

####    a. La arquitectura de la red será igual a la parte encoder del autencoder definido en los ejercicios anteriores
####    b. El modelo debe entrenar correctamente.

In [ ]:
data_augmentation = tf.keras.models.Sequential(
    [
        tf.keras.layers.Input(shape=(32, 32, 3)),
        # tf.keras.layers.RandomFlip("horizontal"),  # Puede ser util en otros casos
        tf.keras.layers.RandomRotation(0.05),
        tf.keras.layers.RandomTranslation(0.15, 0.15),
        tf.keras.layers.RandomZoom(.15),
    ]
)

data_augmentation_2 = tf.keras.models.Sequential(
    [
        tf.keras.layers.Input(shape=(32, 32, 3)),
        # tf.keras.layers.RandomFlip("horizontal"),  # Puede ser util en otros casos
        tf.keras.layers.RandomTranslation(0.15, 0.15),
        tf.keras.layers.Resizing(48, 48), # para CIFAR, para MNIST usar 40 en lugar de 48
        tf.keras.layers.RandomCrop(32, 32), # para CIFAR, para MNIST usar 28 en lugar de 32
    ]
)

In [ ]:
class ModeloAug:

    def __init__(self, tau=5.0, lamda=0.5):
        self.tau = tau
        self.lamda = lamda
        self.aug = data_augmentation
        self.aug2 = data_augmentation_2


        inputs = tf.keras.layers.Input(shape=(32, 32, 3))
        # encoder
        x = tf.keras.layers.Conv2D(64, (3, 3), activation="relu", padding='same')(inputs)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Conv2D(64, (3, 3), activation="relu", padding='same')(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(x)
        x = tf.keras.layers.Dropout(0.2)(x)

        x = tf.keras.layers.Conv2D(128, (3, 3), activation="relu", padding='same')(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Conv2D(128, (3, 3), activation="relu", padding='same')(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(x)
        x = tf.keras.layers.Dropout(0.3)(x)

        x = tf.keras.layers.Conv2D(256, (3, 3), activation="relu", padding='same')(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(x)

        x = tf.keras.layers.Flatten()(x)
        code = tf.keras.layers.Dense(512, activation="relu")(x)
        cluster = tf.keras.layers.Dense(100, activation="softmax", name="classification")(code)

        self.encoder = tf.keras.Model(inputs, code)
        self.cluster_model = tf.keras.Model(inputs, cluster)
        self.optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001)

    def loss_C(self, clustered1, clustered2):
        return tf.reduce_mean(clustered1*(1-clustered1) + clustered2*(1-clustered2))   # reduce_mean pasa de matriz a escalar

    def loss_M(self, encoded1, encoded2):
        M = tf.matmul(encoded1, encoded2, transpose_b=True)
        identity_matrix = tf.eye((tf.shape(encoded1)[0]))  # matriz identidad
        return tf.reduce_mean(
            tf.keras.losses.categorical_crossentropy(identity_matrix, (tf.nn.softmax(M/self.tau, axis=1)))
        )

    def fit(self, X, epochs=10, batch_size=128):
        dataset = tf.data.Dataset.from_tensor_slices(X).batch(batch_size)
        for epoch in range(epochs):
            for batch in dataset:
                # batch = tf.reshape(batch, (-1, 32, 32, 3))
                with tf.GradientTape() as tape:
                    augX1 = self.aug(batch)
                    augX2 = self.aug2(batch)

                    encoded1 = self.encoder(augX1)
                    encoded2 = self.encoder(augX2)
                    
                    clustered1 = self.cluster_model(augX1)
                    clustered2 = self.cluster_model(augX2)

                    lossC = self.loss_C(clustered1, clustered2)
                    lossM = self.loss_M(encoded1, encoded2)

                    loss = lossM + self.lamda*lossC

                vars = self.cluster_model.trainable_variables    # cogemos solo los pesos del modelo completo (encoder+cluster)
                grads = tape.gradient(loss, vars)     # calculamos los gradientes del loss respecto a cada peso
                self.optimizer.apply_gradients(zip(grads, vars))   # actualizamos pesos
            print("epoch:", (epoch+1), "loss:", float(loss))

    def get_representation(self, X):
        return self.encoder.predict(X)

    def predict_clusters(self, X):
        probs = self.cluster_model.predict(X.reshape(-1, 32, 32, 3))
        return np.argmax(probs, axis=1)

    def __del__(self):
        tf.keras.backend.clear_session()

In [ ]:
model = ModeloAug()
model.fit(x_train, epochs=10, batch_size=128)
clusters = model.predict_clusters(x_test)